<a href="https://www.kaggle.com/code/seba023/database-assignment?scriptVersionId=315194136" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd

#paths from scan
movies = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')

# See the columns
print("=== MOVIES columns ===")
print(movies.columns.tolist())

print("\n=== CREDITS columns ===")
print(credits.columns.tolist())

# Peek at the  JSON columns
print("\n=== Example genres cell ===")
print(movies['genres'][0])

print("\n=== Example cast cell ===")
print(credits['cast'][0])

=== MOVIES columns ===
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

=== CREDITS columns ===
['movie_id', 'title', 'cast', 'crew']

=== Example genres cell ===
[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]

=== Example cast cell ===
[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "

## Entity-Relationship Diagram

### Normalization Process
- **1NF:** Split JSON columns (genres, cast, keywords) into separate rows and tables
- **2NF:** Movie details moved to their own table to avoid repetition
- **3NF:** Each concept gets its own table — genres, people, keywords, companies
- **4NF:** Independent lists (cast vs genres) kept in completely separate junction tables

### ER Diagram
![ER Diagram](https://i.imgur.com/AHIEO3N.png)

In [2]:
import pandas as pd
import json
import sqlite3

# load the CSV files
movies = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/datasets/organizations/tmdb/tmdb-movie-metadata/tmdb_5000_credits.csv')

#check total rows available
print(f"Movies rows: {len(movies)}")
print(f"Credits rows: {len(credits)}")
print("\nMovies columns:", movies.columns.tolist())
print("\nCredits columns:", credits.columns.tolist())

Movies rows: 4803
Credits rows: 4803

Movies columns: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

Credits columns: ['movie_id', 'title', 'cast', 'crew']


In [3]:
# ──> table 1: movies
movies_table = movies[[
    'id', 'title', 'budget', 'revenue', 'popularity',
    'vote_average', 'vote_count', 'runtime', 'status',
    'original_language', 'release_date', 'tagline',
    'overview', 'homepage'
]].copy()

# renaming id to be explicit
movies_table = movies_table.rename(columns={'id': 'movie_id'})

# droping duplicate movies 
movies_table = movies_table.drop_duplicates(subset='movie_id')

print(f"Movies table rows: {len(movies_table)}")
print(movies_table.head(3))

Movies table rows: 4803
   movie_id                                     title     budget     revenue  \
0     19995                                    Avatar  237000000  2787965087   
1       285  Pirates of the Caribbean: At World's End  300000000   961000000   
2    206647                                   Spectre  245000000   880674609   

   popularity  vote_average  vote_count  runtime    status original_language  \
0  150.437577           7.2       11800    162.0  Released                en   
1  139.082615           6.9        4500    169.0  Released                en   
2  107.376788           6.3        4466    148.0  Released                en   

  release_date                                         tagline  \
0   2009-12-10                     Enter the World of Pandora.   
1   2007-05-19  At the end of the world, the adventure begins.   
2   2015-10-26                           A Plan No One Escapes   

                                            overview  \
0  In the 22n

In [4]:
# ──> tables 2 & 3: genres and movie_genres 

genres_list = []
movie_genres_list = []

for _, row in movies.iterrows():
    try:
        genres = json.loads(row['genres'])
        for g in genres:
            # genres table — unique genres only
            genres_list.append({'genre_id': g['id'], 'name': g['name']})
            # movie_genres table — which movie has which genre
            movie_genres_list.append({'movie_id': row['id'], 'genre_id': g['id']})
    except:
        pass

# creating dataframes and removing duplicates
genres_table = pd.DataFrame(genres_list).drop_duplicates(subset='genre_id')
movie_genres_table = pd.DataFrame(movie_genres_list).drop_duplicates()

print(f"Genres table rows: {len(genres_table)}")
print(genres_table.head(5))
print(f"\nMovie_genres table rows: {len(movie_genres_table)}")
print(movie_genres_table.head(5))

Genres table rows: 20
   genre_id             name
0        28           Action
1        12        Adventure
2        14          Fantasy
3       878  Science Fiction
9        80            Crime

Movie_genres table rows: 12160
   movie_id  genre_id
0     19995        28
1     19995        12
2     19995        14
3     19995       878
4       285        12


In [5]:
# ──> table 4 & 5: keywords and movie_keywords
keywords_list = []
movie_keywords_list = []

for _, row in movies.iterrows():
    try:
        keywords = json.loads(row['keywords'])
        for k in keywords:
            # keywords table — unique keywords only
            keywords_list.append({'keyword_id': k['id'], 'name': k['name']})
            # movie_keywords table — which movie has which keyword
            movie_keywords_list.append({'movie_id': row['id'], 'keyword_id': k['id']})
    except:
        pass

# creating dataframes + removing potential duplicates
keywords_table = pd.DataFrame(keywords_list).drop_duplicates(subset='keyword_id')
movie_keywords_table = pd.DataFrame(movie_keywords_list).drop_duplicates()

print(f"Keywords table rows: {len(keywords_table)}")
print(keywords_table.head(5))
print(f"\nMovie_keywords table rows: {len(movie_keywords_table)}")
print(movie_keywords_table.head(5))

Keywords table rows: 9813
   keyword_id           name
0        1463  culture clash
1        2964         future
2        3386      space war
3        3388   space colony
4        3679        society

Movie_keywords table rows: 36194
   movie_id  keyword_id
0     19995        1463
1     19995        2964
2     19995        3386
3     19995        3388
4     19995        3679


In [6]:
# ──>table 6 & 7: production_companes and movie_companies
companies_list = []
movie_companies_list = []

for _, row in movies.iterrows():
    try:
        companies = json.loads(row['production_companies'])
        for c in companies:
            # production_companies table — unique companies only
            companies_list.append({'company_id': c['id'], 'name': c['name']})
            # movie_companies table — which movie has which company
            movie_companies_list.append({'movie_id': row['id'], 'company_id': c['id']})
    except:
        pass

# creating dataframes + removing duplicates
companies_table = pd.DataFrame(companies_list).drop_duplicates(subset='company_id')
movie_companies_table = pd.DataFrame(movie_companies_list).drop_duplicates()

print(f"Companies table rows: {len(companies_table)}")
print(companies_table.head(5))
print(f"\nMovie_companies table rows: {len(movie_companies_table)}")
print(movie_companies_table.head(5))

Companies table rows: 5047
   company_id                                    name
0         289                 Ingenious Film Partners
1         306  Twentieth Century Fox Film Corporation
2         444                      Dune Entertainment
3         574                Lightstorm Entertainment
4           2                    Walt Disney Pictures

Movie_companies table rows: 13677
   movie_id  company_id
0     19995         289
1     19995         306
2     19995         444
3     19995         574
4       285           2


In [7]:
# ──>able 8 & 9: people and movie_cast
people_list = []
movie_cast_list = []

for _, row in credits.iterrows():
    try:
        cast = json.loads(row['cast'])
        for c in cast:
            # people table — unique people only
            people_list.append({
                'person_id': c['id'],
                'name': c['name'],
                'gender': c['gender']
            })
            # movie_cast table — who played what in which movie
            movie_cast_list.append({
                'movie_id': row['movie_id'],
                'person_id': c['id'],
                'character': c['character'],
                'cast_order': c['order']
            })
    except:
        pass

# creating dataframes + removing duplicates
people_table = pd.DataFrame(people_list).drop_duplicates(subset='person_id')
movie_cast_table = pd.DataFrame(movie_cast_list).drop_duplicates()

print(f"People table rows: {len(people_table)}")
print(people_table.head(5))
print(f"\nMovie_cast table rows: {len(movie_cast_table)}")
print(movie_cast_table.head(5))

People table rows: 54588
   person_id                name  gender
0      65731     Sam Worthington       2
1       8691         Zoe Saldana       1
2      10205    Sigourney Weaver       1
3      32747        Stephen Lang       2
4      17647  Michelle Rodriguez       1

Movie_cast table rows: 106257
   movie_id  person_id            character  cast_order
0     19995      65731           Jake Sully           0
1     19995       8691              Neytiri           1
2     19995      10205  Dr. Grace Augustine           2
3     19995      32747        Col. Quaritch           3
4     19995      17647         Trudy Chacon           4


In [8]:
# ──> creqting SQLlite database

conn = sqlite3.connect('/kaggle/working/tmdb_movies.db')
cursor = conn.cursor()


# ──> LOAD ALL TABLES INTO THE DATABASE 

# main Movies table
movies_table.to_sql('movies', conn, if_exists='replace', index=False)

# Genres
genres_table.to_sql('genres', conn, if_exists='replace', index=False)
movie_genres_table.to_sql('movie_genres', conn, if_exists='replace', index=False)

# Keywords
keywords_table.to_sql('keywords', conn, if_exists='replace', index=False)
movie_keywords_table.to_sql('movie_keywords', conn, if_exists='replace', index=False)

# Companies
companies_table.to_sql('production_companies', conn, if_exists='replace', index=False)
movie_companies_table.to_sql('movie_companies', conn, if_exists='replace', index=False)

# People and cast
people_table.to_sql('people', conn, if_exists='replace', index=False)
movie_cast_table.to_sql('movie_cast', conn, if_exists='replace', index=False)


# ──> verifying table creation
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tables created in database:")
for t in tables:
    print(f"  ✓ {t[0]}")

conn.commit()
print("\nDatabase saved successfully at /kaggle/working/tmdb_movies.db")

Tables created in database:
  ✓ movies
  ✓ genres
  ✓ movie_genres
  ✓ keywords
  ✓ movie_keywords
  ✓ production_companies
  ✓ movie_companies
  ✓ people
  ✓ movie_cast

Database saved successfully at /kaggle/working/tmdb_movies.db


In [9]:
# ──> DEFINING PRIMARY KEYS AND FOREIGN KEYS 

cursor.executescript('''
    CREATE TABLE IF NOT EXISTS movies_clean (
        movie_id    INTEGER PRIMARY KEY,
        title       TEXT,
        budget      REAL,
        revenue     REAL,
        popularity  REAL,
        vote_average REAL,
        vote_count  INTEGER,
        runtime     REAL,
        status      TEXT,
        original_language TEXT,
        release_date TEXT,
        tagline     TEXT,
        overview    TEXT,
        homepage    TEXT
    );

    CREATE TABLE IF NOT EXISTS genres_clean (
        genre_id    INTEGER PRIMARY KEY,
        name        TEXT
    );

    CREATE TABLE IF NOT EXISTS movie_genres_clean (
        movie_id    INTEGER REFERENCES movies_clean(movie_id),
        genre_id    INTEGER REFERENCES genres_clean(genre_id)
    );

    CREATE TABLE IF NOT EXISTS keywords_clean (
        keyword_id  INTEGER PRIMARY KEY,
        name        TEXT
    );

    CREATE TABLE IF NOT EXISTS movie_keywords_clean (
        movie_id    INTEGER REFERENCES movies_clean(movie_id),
        keyword_id  INTEGER REFERENCES keywords_clean(keyword_id)
    );

    CREATE TABLE IF NOT EXISTS production_companies_clean (
        company_id  INTEGER PRIMARY KEY,
        name        TEXT
    );

    CREATE TABLE IF NOT EXISTS movie_companies_clean (
        movie_id    INTEGER REFERENCES movies_clean(movie_id),
        company_id  INTEGER REFERENCES production_companies_clean(company_id)
    );

    CREATE TABLE IF NOT EXISTS people_clean (
        person_id   INTEGER PRIMARY KEY,
        name        TEXT,
        gender      INTEGER
    );

    CREATE TABLE IF NOT EXISTS movie_cast_clean (
        movie_id    INTEGER REFERENCES movies_clean(movie_id),
        person_id   INTEGER REFERENCES people_clean(person_id),
        character   TEXT,
        cast_order  INTEGER
    );
''')

conn.commit()
print("✅ All Primary Keys and Foreign Keys defined successfully!")

✅ All Primary Keys and Foreign Keys defined successfully!
